# Horn of Africa Situation Room — Notebook 01

This notebook runs the full pipeline:

1. Load raw datasets (`data/raw/*.csv`)
2. Clean + join with location metadata
3. Compute **regional risk scores** with a **cross‑border refugee pressure index** (South Sudan → Gambella)
4. Produce charts (saved to `outputs/figures/`)
5. Generate a weekly diplomatic brief (saved to `outputs/briefs/`)

> Designed to be interpretable and for policy makers and NGO professionals.


## 0) Setup

Run this once per environment.


In [53]:
gitignore_content = """
__pycache__/
*.pyc
.ipynb_checkpoints/

.env

data/raw/*
data/clean/*

outputs/
node_modules/
.next/

.DS_Store
Thumbs.db
"""

with open(".gitignore", "w") as f:
    f.write(gitignore_content)

print("✅ .gitignore created")

✅ .gitignore created


In [54]:
import sys
from pathlib import Path

# Get project root (one level above notebooks folder)
PROJECT_ROOT = Path().resolve().parents[0]

# Add project root to Python path
sys.path.append(str(PROJECT_ROOT))

print("Project root added to path:", PROJECT_ROOT)

Project root added to path: C:\Users\getab\OneDrive\Documents\projects\horn-of-afica-situation-room


In [55]:
# If you haven't installed requirements yet, run:
# !pip install -r ../requirements.txt

import pandas as pd
from pathlib import Path

# Make sure plots appear inline
%matplotlib inline


In [56]:
# Make sure thate pandas is installed
# pip install pandas

## 1) Paths and folders

In [57]:
from src.io_utils import ensure_dirs, PROJECT_ROOT

raw_dir = PROJECT_ROOT / "data" / "raw"
clean_dir = PROJECT_ROOT / "data" / "clean"
out_fig = PROJECT_ROOT / "outputs" / "figures"
out_tables = PROJECT_ROOT / "outputs" / "tables"
out_briefs = PROJECT_ROOT / "outputs" / "briefs"

ensure_dirs(raw_dir, clean_dir, out_fig, out_tables, out_briefs)

raw_dir, out_fig, out_tables, out_briefs


(WindowsPath('C:/Users/getab/OneDrive/Documents/projects/horn-of-afica-situation-room/data/raw'),
 WindowsPath('C:/Users/getab/OneDrive/Documents/projects/horn-of-afica-situation-room/outputs/figures'),
 WindowsPath('C:/Users/getab/OneDrive/Documents/projects/horn-of-afica-situation-room/outputs/tables'),
 WindowsPath('C:/Users/getab/OneDrive/Documents/projects/horn-of-afica-situation-room/outputs/briefs'))

## 2) Load raw data

Expected raw files:

- `locations.csv`
- `incidents.csv`
- `displacement.csv` 


In [58]:
locations_path = raw_dir / "locations.csv"
incidents_path = raw_dir / "incidents.csv"
displacement_path = raw_dir / "displacement.csv"

locations = pd.read_csv(locations_path)
incidents_raw = pd.read_csv(incidents_path)
displacement_raw = pd.read_csv(displacement_path)

print("locations:", locations.shape)
print("incidents_raw:", incidents_raw.shape)
print("displacement_raw:", displacement_raw.shape)

display(locations.head(5))
display(incidents_raw.head(5))
display(displacement_raw.head(5))


locations: (8, 7)
incidents_raw: (6, 8)
displacement_raw: (46, 8)


,location_id,country,admin1,admin2,region,latitude,longitude
0,LOC001,Ethiopia,Gambella,NaN,Gambella,8.25,34.58
1,LOC002,Sudan,Khartoum,NaN,Khartoum,15.50,32.56
2,LOC003,Somalia,Banadir,Mogadishu,Banadir,2.04,45.34
3,LOC004,Kenya,Nairobi,Nairobi,Nairobi,-1.29,36.82
4,LOC005,South Sudan,Upper Nile,Malakal,Upper Nile,9.54,31.66


,event_id,event_date,location_id,event_type,severity,reported_fatalities,source,notes
0,EVT1001,2026-02-25,LOC001,Armed Clash,4,7,Media,Clashes reported near border area
1,EVT1002,2026-02-26,LOC003,Protest,2,0,NGO,Demonstration with arrests reported
2,EVT1003,2026-02-27,LOC002,Explosion,5,12,Media,Explosion in urban area
3,EVT1004,2026-02-20,LOC001,Abduction,3,0,Community,Local reports of abduction
4,EVT1005,2026-02-24,LOC005,Inter-communal Violence,4,15,Media,Fighting reported in Upper Nile


,month,location_id,displacement_type,origin_country,arrivals,departures,source,notes
0,2025-07,LOC005,IDP,NaN,1800,250,UN,Inter-communal violence
1,2025-08,LOC005,IDP,NaN,2000,300,UN,Escalation in Upper Nile
2,2025-09,LOC005,IDP,NaN,2100,350,UN,Clashes reported
3,2025-10,LOC005,IDP,NaN,2200,300,UN,Internal displacement in Upper Nile
4,2025-11,LOC005,IDP,NaN,2800,450,UN,Continued insecurity


## 3) Clean data + join locations

In [59]:
from src.cleaning import clean_incidents, clean_displacement, join_locations

incidents = clean_incidents(incidents_raw)
displacement = clean_displacement(displacement_raw)

inc_geo = join_locations(incidents, locations)
disp_geo = join_locations(displacement, locations)

print("incidents cleaned:", incidents.shape)
print("displacement cleaned:", displacement.shape)
print("inc_geo:", inc_geo.shape)
print("disp_geo:", disp_geo.shape)

display(inc_geo.head(5))
display(disp_geo.head(5))


incidents cleaned: (6, 8)
displacement cleaned: (46, 8)
inc_geo: (6, 14)
disp_geo: (46, 14)


,event_id,event_date,location_id,event_type,severity,reported_fatalities,source,notes,country,admin1,admin2,region,latitude,longitude
0,EVT1001,2026-02-25,LOC001,Armed Clash,4,7,Media,Clashes reported near border area,Ethiopia,Gambella,NaN,Gambella,8.25,34.58
1,EVT1002,2026-02-26,LOC003,Protest,2,0,NGO,Demonstration with arrests reported,Somalia,Banadir,Mogadishu,Banadir,2.04,45.34
2,EVT1003,2026-02-27,LOC002,Explosion,5,12,Media,Explosion in urban area,Sudan,Khartoum,NaN,Khartoum,15.50,32.56
3,EVT1004,2026-02-20,LOC001,Abduction,3,0,Community,Local reports of abduction,Ethiopia,Gambella,NaN,Gambella,8.25,34.58
4,EVT1005,2026-02-24,LOC005,Inter-communal Violence,4,15,Media,Fighting reported in Upper Nile,South Sudan,Upper Nile,Malakal,Upper Nile,9.54,31.66


,month,location_id,displacement_type,origin_country,arrivals,departures,source,notes,country,admin1,admin2,region,latitude,longitude
0,2025-07-01,LOC005,IDP,,1800,250,UN,Inter-communal violence,South Sudan,Upper Nile,Malakal,Upper Nile,9.54,31.66
1,2025-08-01,LOC005,IDP,,2000,300,UN,Escalation in Upper Nile,South Sudan,Upper Nile,Malakal,Upper Nile,9.54,31.66
2,2025-09-01,LOC005,IDP,,2100,350,UN,Clashes reported,South Sudan,Upper Nile,Malakal,Upper Nile,9.54,31.66
3,2025-10-01,LOC005,IDP,,2200,300,UN,Internal displacement in Upper Nile,South Sudan,Upper Nile,Malakal,Upper Nile,9.54,31.66
4,2025-11-01,LOC005,IDP,,2800,450,UN,Continued insecurity,South Sudan,Upper Nile,Malakal,Upper Nile,9.54,31.66


## 4) Validate key assumptions 

- `event_date` must parse correctly  
- `month` must parse correctly  
- Gambella locations should have `region == Gambella`  
- Refugee rows should have `origin_country` (for cross-border pressure)


In [60]:
# Basic QA checks
qa = {}

qa["incidents_missing_event_date"] = int(incidents["event_date"].isna().sum())
qa["displacement_missing_month"] = int(displacement["month"].isna().sum())

# Count refugee rows missing origin_country
ref_rows = disp_geo[disp_geo["displacement_type"].str.lower() == "refugee"].copy()
qa["refugee_rows_total"] = int(len(ref_rows))
qa["refugee_rows_missing_origin_country"] = int((ref_rows["origin_country"].fillna("").str.strip() == "").sum())

# Gambella check
gambella_rows = disp_geo[(disp_geo["country"] == "Ethiopia") & (disp_geo["region"] == "Gambella")]
qa["gambella_rows"] = int(len(gambella_rows))

qa


{'incidents_missing_event_date': 0,
 'displacement_missing_month': 0,
 'refugee_rows_total': 19,
 'refugee_rows_missing_origin_country': 0,
 'gambella_rows': 22}

## 5) Compute South Sudan → Gambella refugee pressure index

This creates a **0–100 index** based on rolling (3-month) arrivals.


In [61]:
from src.refugee_pressure import compute_refugee_pressure_index

pressure_ts = compute_refugee_pressure_index(
    disp_geo,
    target_country="Ethiopia",
    target_region="Gambella",
    origin_country="South Sudan",
    window_months=3,
)

display(pressure_ts.tail(12))

pressure_latest = float(pressure_ts["pressure_index_0_100"].iloc[-1]) if not pressure_ts.empty else 0.0
pressure_latest


,month,rolling_arrivals,pressure_index_0_100
0,2025-07-01,600.0,5.172414
1,2025-08-01,1350.0,11.637931
2,2025-09-01,2670.0,23.017241
3,2025-10-01,3620.0,31.206897
4,2025-11-01,4820.0,41.551724
5,2025-12-01,6000.0,51.724138
6,2026-01-01,7550.0,65.086207
7,2026-02-01,9400.0,81.034483
8,2026-03-01,11600.0,100.000000


100.0

Save pressure time series.

In [62]:
pressure_out = out_tables / "ssudan_to_gambella_pressure.csv"
pressure_ts.to_csv(pressure_out, index=False)
pressure_out


WindowsPath('C:/Users/getab/OneDrive/Documents/projects/horn-of-afica-situation-room/outputs/tables/ssudan_to_gambella_pressure.csv')

## 6) Compute regional risk scores (with refugee pressure blended into Gambella)

**Interpretation**  
- Every region gets an incident-based score (0–100)  
- **Gambella (Ethiopia)** also gets a blended score that includes the refugee pressure index  


In [63]:
from src.risk_scoring import compute_region_risk

week_end = pd.to_datetime("2026-03-01")  # change as needed

risk_table = compute_region_risk(
    inc_geo,
    week_end,
    refugee_pressure_latest_0_100=pressure_latest,
    refugee_pressure_weight=0.25,  # recommended default
    refugee_pressure_target_country="Ethiopia",
    refugee_pressure_target_region="Gambella",
)

display(risk_table.head(15))

risk_out = out_tables / "risk_scores.csv"
risk_table.to_csv(risk_out, index=False)
risk_out


,country,region,incident_count,avg_severity,fatalities,incident_score_raw,last_event,incident_score_0_100,refugee_pressure_0_100,risk_score_0_100,risk_level
0,Ethiopia,Gambella,2,3.5,7,6.839721,2026-02-25,100.000000,100.0,100.000000,HIGH
2,South Sudan,Jonglei,1,5.0,22,6.567747,2026-02-22,96.023615,0.0,96.023615,HIGH
4,Sudan,Khartoum,1,5.0,12,6.282475,2026-02-27,91.852795,0.0,91.852795,HIGH
3,South Sudan,Upper Nile,1,4.0,15,5.386294,2026-02-24,78.750208,0.0,78.750208,HIGH
1,Somalia,Banadir,1,2.0,0,2.000000,2026-02-26,29.240960,0.0,29.240960,LOW


WindowsPath('C:/Users/getab/OneDrive/Documents/projects/horn-of-afica-situation-room/outputs/tables/risk_scores.csv')

## 7) Forecast displacement (baseline)

A simple linear trend on net flows (arrivals - departures).  
Designed to be easy to explain in a policy brief.


In [64]:
from src.displacement_forecast import forecast_displacement

forecast = forecast_displacement(disp_geo, months_ahead=6)
display(forecast.head(20))

forecast_out = out_tables / "displacement_forecast.csv"
if not forecast.empty:
    forecast.to_csv(forecast_out, index=False)
forecast_out


,country,region,displacement_type,month,net_flow_forecast
0,Ethiopia,Gambella,IDP,2026-04-01,1741
1,Ethiopia,Gambella,IDP,2026-05-01,1905
2,Ethiopia,Gambella,IDP,2026-06-01,2069
3,Ethiopia,Gambella,IDP,2026-07-01,2233
4,Ethiopia,Gambella,IDP,2026-08-01,2397
5,Ethiopia,Gambella,IDP,2026-09-01,2561
6,Ethiopia,Gambella,Refugee,2026-04-01,1995
7,Ethiopia,Gambella,Refugee,2026-05-01,2100
8,Ethiopia,Gambella,Refugee,2026-06-01,2204
9,Ethiopia,Gambella,Refugee,2026-07-01,2309


WindowsPath('C:/Users/getab/OneDrive/Documents/projects/horn-of-afica-situation-room/outputs/tables/displacement_forecast.csv')

## 8) Visualizations

This will save PNGs into `outputs/figures/`:
- Incidents daily trend (last 30 days)
- Displacement history + forecast
- South Sudan → Gambella arrivals


In [65]:
from src.plots import (
    plot_incidents_timeseries,
    plot_displacement_forecast,
    plot_ssudan_to_gambella_flow,
)

plot_incidents_timeseries(incidents, out_fig / "incidents_last_30_days.png", days=30)
plot_displacement_forecast(disp_geo, forecast, out_fig / "displacement_forecast.png")
plot_ssudan_to_gambella_flow(
    disp_geo,
    out_fig / "ssudan_to_gambella_arrivals.png",
    target_country="Ethiopia",
    target_region="Gambella",
    origin_country="South Sudan",
)

list(out_fig.glob("*.png"))


[WindowsPath('C:/Users/getab/OneDrive/Documents/projects/horn-of-afica-situation-room/outputs/figures/displacement_forecast.png'),
 WindowsPath('C:/Users/getab/OneDrive/Documents/projects/horn-of-afica-situation-room/outputs/figures/incidents_last_30_days.png'),
 WindowsPath('C:/Users/getab/OneDrive/Documents/projects/horn-of-afica-situation-room/outputs/figures/ssudan_to_gambella_arrivals.png')]

## 9) Weekly brief 

Saved into `outputs/briefs/`.


In [66]:
from src.brief_generator import generate_weekly_brief

brief_text = generate_weekly_brief(
    week_end=week_end,
    risk_table=risk_table,
    incidents=incidents,
    forecast=forecast,
)

brief_path = out_briefs / f"weekly_brief_{week_end.date()}.md"
brief_path.write_text(brief_text, encoding="utf-8")

brief_path


WindowsPath('C:/Users/getab/OneDrive/Documents/projects/horn-of-afica-situation-room/outputs/briefs/weekly_brief_2026-03-01.md')

Preview brief:

In [67]:
print(brief_text[:2000])  # show first part of brief


# Weekly Situation Brief — Horn of Africa
**Week ending:** 2026-03-01

## Executive Summary
- Total incidents in dataset: **6**
- Incidents last 7 days: **5**
- Incidents last 30 days: **6**

## Top Risk Areas (Region-level)

| Rank | Country | Region | Risk Level | Score (0–100) | Incidents | Avg Severity | Last Event |
|---:|---|---|---|---:|---:|---:|---|
| 1 | Ethiopia | Gambella | HIGH | 100.0 | 2 | 3.50 | 2026-02-25 |
| 2 | South Sudan | Jonglei | HIGH | 96.0 | 1 | 5.00 | 2026-02-22 |
| 3 | Sudan | Khartoum | HIGH | 91.9 | 1 | 5.00 | 2026-02-27 |
| 4 | South Sudan | Upper Nile | HIGH | 78.8 | 1 | 4.00 | 2026-02-24 |
| 5 | Somalia | Banadir | LOW | 29.2 | 1 | 2.00 | 2026-02-26 |

## Displacement Outlook (Baseline Forecast)
- Projected net displacement (next 3 months, total across regions):
  - 2026-01: **1,027**
  - 2026-02: **1,849**
  - 2026-03: **2,129**

## Analyst Notes (Fill-in)
- Key drivers (security, politics, economy, climate):
- Diplomatic engagement opportunities / ris

# Weekly Situation Brief — Horn of Africa
**Week ending:** 2026-03-01

## Executive Summary
- Total incidents in dataset: **6**
- Incidents last 7 days: **5**
- Incidents last 30 days: **6**

## Top Risk Areas (Region-level)

| Rank | Country | Region | Risk Level | Score (0–100) | Incidents | Avg Severity | Last Event |
|---:|---|---|---|---:|---:|---:|---|
| 1 | Ethiopia | Gambella | HIGH | 100.0 | 2 | 3.50 | 2026-02-25 |
| 2 | South Sudan | Jonglei | HIGH | 96.0 | 1 | 5.00 | 2026-02-22 |
| 3 | Sudan | Khartoum | HIGH | 91.9 | 1 | 5.00 | 2026-02-27 |
| 4 | South Sudan | Upper Nile | HIGH | 78.8 | 1 | 4.00 | 2026-02-24 |
| 5 | Somalia | Banadir | LOW | 29.2 | 1 | 2.00 | 2026-02-26 |

## Displacement Outlook (Baseline Forecast)
- Projected net displacement (next 3 months, total across regions):
  - 2026-01: **1,027**
  - 2026-02: **1,849**
  - 2026-03: **2,129**

## Analyst Notes (Fill-in)
- Key drivers (security, politics, economy, climate):
- Diplomatic engagement opportunities / risks:
- Recommended actions (7–14 days):

## 10) Next

- Lag correlation: South Sudan incidents → Gambella arrivals after 1–2 months  
- Camp capacity stress index (arrivals vs capacity)  
- Confidence weighting by source type  
- Interactive map (Folium)  
